# Using Model Library to Rerank the Search Results

使用 PyMilvus 的重排序模型（2.4.0+ 版本）提升您的搜索结果，这些模型旨在优化并优先展示最相关的搜索结果。本指南将为您介绍如何集成先进的重排序模型，以增强搜索系统的准确性。首先快速设置必要的依赖项（可选择在虚拟环境中），然后详细介绍如何利用这些模型，将您的搜索功能提升至全新水平。

In [1]:
# ! pip install pymilvus==2.4.0
# ! pip install "pymilvus[model]"

In [2]:
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'

os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [3]:
query = "What event in 1956 marked the official birth of artificial intelligence as a discipline?"
documents = [
    "In 1950, Alan Turing published his seminal paper, 'Computing Machinery and Intelligence,' proposing the Turing Test as a criterion of intelligence, a foundational concept in the philosophy and development of artificial intelligence.",
    "The Dartmouth Conference in 1956 is considered the birthplace of artificial intelligence as a field; here, John McCarthy and others coined the term 'artificial intelligence' and laid out its basic goals.",
    "In 1951, British mathematician and computer scientist Alan Turing also developed the first program designed to play chess, demonstrating an early example of AI in game strategy.",
    "The invention of the Logic Theorist by Allen Newell, Herbert A. Simon, and Cliff Shaw in 1955 marked the creation of the first true AI program, which was capable of solving logic problems, akin to proving mathematical theorems."
]

## BGE Rerank Function

此重排功能使用 FlagEmbedding 的重排器，根据查询对文档进行重新排序。欲了解更多信息，请访问其代码仓库。

transformers 5.x 不兼容 FlagEmbedding，需要补丁

In [4]:
from pymilvus.model.reranker import BGERerankFunction

# transformers v5 移除了 tokenizer.prepare_for_model,而 FlagEmbedding 仍依赖它。
# 不改动任何包文件:运行时给 tokenizer 实例补上该方法(用 v5 仍存在的 API 实现)。
def _attach_prepare_for_model(tokenizer):
    def prepare_for_model(ids_0, ids_1=None, truncation=None, max_length=None, padding=False, **kwargs):
        if ids_1 is not None:
            n_special = tokenizer.num_special_tokens_to_add(pair=True)
            if max_length is not None:
                over = len(ids_0) + len(ids_1) + n_special - max_length
                if over > 0:
                    if truncation == 'only_second':
                        ids_1 = ids_1[:max(0, len(ids_1) - over)]
                    else:
                        ids_0 = ids_0[:max(0, len(ids_0) - over)]
                        ids_1 = ids_1[:max(0, len(ids_1) - over)]
            input_ids = ([tokenizer.cls_token_id] + list(ids_0)
                         + [tokenizer.sep_token_id, tokenizer.sep_token_id]
                         + list(ids_1) + [tokenizer.sep_token_id])
        else:
            n_special = tokenizer.num_special_tokens_to_add(pair=False)
            if max_length is not None and len(ids_0) + n_special > max_length:
                ids_0 = ids_0[:max_length - n_special]
            input_ids = [tokenizer.cls_token_id] + list(ids_0) + [tokenizer.sep_token_id]
        out = {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids)}
        if padding:
            padded = tokenizer.pad([out], padding=True, return_tensors='pt')
            out = {k: v[0] for k, v in padded.items()}
        return out
    tokenizer.prepare_for_model = prepare_for_model

In [5]:
bge_rf=BGERerankFunction(device='cuda:0')
_attach_prepare_for_model(bge_rf.reranker.tokenizer)

results=bge_rf(query,documents,top_k=3)

print(len(results),results)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

3 [RerankResult(text="The Dartmouth Conference in 1956 is considered the birthplace of artificial intelligence as a field; here, John McCarthy and others coined the term 'artificial intelligence' and laid out its basic goals.", score=0.991186833496954, index=1), RerankResult(text="In 1950, Alan Turing published his seminal paper, 'Computing Machinery and Intelligence,' proposing the Turing Test as a criterion of intelligence, a foundational concept in the philosophy and development of artificial intelligence.", score=0.03277498389995642, index=0), RerankResult(text='The invention of the Logic Theorist by Allen Newell, Herbert A. Simon, and Cliff Shaw in 1955 marked the creation of the first true AI program, which was capable of solving logic problems, akin to proving mathematical theorems.', score=0.006488269596656705, index=3)]


## Voyage Rerank Function

此重排功能使用 Voyage AI 的重排器，根据查询重新排序文档。欲了解更多信息，请访问其文档页面。

需要api-key

In [5]:
# from pymilvus.model.reranker import VoyageRerankFunction
#
# voyage_rf=VoyageRerankFunction(api_key="voyage-api-key")
# results=voyage_rf(query,documents,top_k=3)
# print(len(results),results)

## Cohere Rerank Function

This rerank function uses Cohere's reranker to reorder documents according to query. For more information, please visit their documentation.

需要api-key

In [6]:
# from pymilvus.model.reranker import CohereRerankFunction

# cohere_rf = CohereRerankFunction(api_key="cohere-api-key")
# results = cohere_rf(query, documents, top_k=3)
# print(len(results), results)

## Cross-Encoder Rerank Function

This rerank function uses Sentence-Transformers's Cross-Encoders to reorder documents according to query. For more information, please visit their documentation.

In [7]:
from pymilvus.model.reranker import CrossEncoderRerankFunction

ce_rf=CrossEncoderRerankFunction(
    model_name_or_path="cross-encoder/ms-marco-MiniLM-L-6-v2",
    device='cuda:0',
)
results=ce_rf(query,documents,top_k=3)
print(len(results),results)

The CrossEncoder `model_name` argument was renamed and is now deprecated. Please use `model_name_or_path` instead.
The CrossEncoder `default_activation_function` argument was renamed and is now deprecated. Please use `activation_fn` instead.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

3 [RerankResult(text="The Dartmouth Conference in 1956 is considered the birthplace of artificial intelligence as a field; here, John McCarthy and others coined the term 'artificial intelligence' and laid out its basic goals.", score=6.250531196594238, index=1), RerankResult(text="In 1950, Alan Turing published his seminal paper, 'Computing Machinery and Intelligence,' proposing the Turing Test as a criterion of intelligence, a foundational concept in the philosophy and development of artificial intelligence.", score=-2.954598903656006, index=0), RerankResult(text='The invention of the Logic Theorist by Allen Newell, Herbert A. Simon, and Cliff Shaw in 1955 marked the creation of the first true AI program, which was capable of solving logic problems, akin to proving mathematical theorems.', score=-4.771507740020752, index=3)]
